# 01 Setup + SFT (Tasks 1 & 3)

In [ ]:
REPO_URL = "https://github.com/NifraWahaj/cs5323_PA2.git"
BRANCH = "main"

import os, subprocess, sys
REPO = "/kaggle/working/sprout"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs visible:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"

## Public adapter tests + edge-case tests 

In [ ]:
!PYTHONPATH=. python -m pytest -q tests

## Freeze the fixed AlpacaEval subset now (300 prompts, seed 42) 

In [ ]:
from sproutlm.generate import ensure_alpaca_subset
import json, shutil, os
p = ensure_alpaca_subset()
os.makedirs("/kaggle/working/outputs/eval_sets", exist_ok=True)
shutil.copy(p, "/kaggle/working/outputs/eval_sets/")
shutil.copy("eval_sets/hh_rlhf_eval_300_seed42.json", "/kaggle/working/outputs/eval_sets/")
print(len(json.load(open(p))), "alpaca prompts ->", p)

## SFT on both T4s (DDP via torchrun)
Global batch = 2 per GPU × 2 GPUs × 8 accumulation = 32. Override any config key with `key=value`.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True NCCL_P2P_DISABLE=1 TOKENIZERS_PARALLELISM=false torchrun --nproc_per_node 2 -m sproutlm.sft_loop \
    --config configs/sft.yaml output_dir=/kaggle/working/outputs/sft
import os
assert os.path.exists("/kaggle/working/outputs/sft/best/sprout_note.json"), "SFT FAILED - read the log above; do not attach this version to 02"

In [ ]:
!pip install -q matplotlib
!python -m sproutlm.plots --run /kaggle/working/outputs/sft --out /kaggle/working/outputs/plots
from IPython.display import Image; Image("/kaggle/working/outputs/plots/sft.png")

In [ ]:
!cat /kaggle/working/outputs/sft/best/sprout_note.json

In [ ]:
!cd /kaggle/working && zip -qr sft_run_outputs.zip outputs/sft/metrics.jsonl outputs/sft/config_used.yaml outputs/plots outputs/eval_sets && ls -lh /kaggle/working && du -sh /kaggle/working/outputs/*